In [1]:
import requests
import pandas as pd
import numpy as np
import time
import os
from cache import* 
from dotenv import load_dotenv
load_dotenv()

api_key = os.getenv("ALPHAVANTAGE_API_KEY")

   timestamp    open      high     low   close  volume
0 2025-08-21  9046.0  9049.578  9000.0  9049.0   45266
1 2025-08-20  9013.0  9040.000  8972.0  9020.0   38695
2 2025-08-19  9046.0  9071.000  9035.0  9050.0   45341
3 2025-08-18  9040.0  9070.000  9015.0  9041.0   88544
4 2025-08-15  9068.0  9071.000  9020.0  9024.0   38466
arse


In [5]:

symbols = ['SWDA.LON', 'EMIM.LON', 'IBM', 'ULVR.LON', ]  # Example symbols

for symbol in symbols:
    print(f"Processing {symbol}...")

    # ------- ASSET --------
    url = (
        f'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY'
        f'&symbol={symbol}&apikey={api_key}&outputsize=full&datatype=csv'
    )
    asset_df = fetch_csv_cached(url=url, function="time_series_daily", key=symbol, ttl_hours=1000, expected_cols=["timestamp", "open", "high", "low", "close", "volume"]).sort_values('timestamp').set_index('timestamp') 
    # USE LAST 252 TRADING DAYS FOR 1-YEAR VOLATILITY
    asset_df = asset_df.rename(columns={'close': 'asset_close_GBx'})
    # print(f"asset_df:\n{asset_df.head()}")

    # ------- FX --------
    fx_url = (
    f'https://www.alphavantage.co/query?function=FX_DAILY'
    f'&from_symbol=GBP&to_symbol=CHF&apikey={api_key}&outputsize=full&datatype=csv'
    )

    fx_df = fetch_csv_cached(url=fx_url, function="fx_daily", key="GBPCHF", ttl_hours=1000, expected_cols=["timestamp", "open", "high", "low", "close"]).sort_values('timestamp').set_index('timestamp')

    # TAKE ONLY 'CLOSE' COLUMN AND RENAME IT
    fx_rate = fx_df['close'].rename( 'fx_rate')
    # print(f"fx_rate:\n{fx_rate.head()}")

    # ------- MERGE & CALC VOL --------
    merged = asset_df.join(fx_rate, how='inner')
    # print(f"merged:\n{merged.head()}")
    merged['close_chf'] = merged['asset_close_GBx'] *merged['fx_rate']  # Convert to CHF

    # CONTINUE AS BEFORE, BUT ON 'CLOSE_CHF'
    prices = merged['close_chf'].tail(252)  # Last 252 trading days
    # print(f"SWDA.LON Prices in CHF:\n{prices.head()}")
    returns = np.log(prices / prices.shift(1)).dropna()

    # DAILY VOLATILITY
    daily_vol = returns.std()

    # ANNUALIZED VOLATILITY
    annual_vol = daily_vol * np.sqrt(252)

    print(f"{symbol} 1Y annualized volatility: {annual_vol:.2%} (GHF-based)")
    # time.sleep(3)


Processing SWDA.LON...
Using cached SWDA.LON copy
Using cached GBPCHF copy
SWDA.LON 1Y annualized volatility: 17.17% (GHF-based)
Processing EMIM.LON...
Using cached EMIM.LON copy
Using cached GBPCHF copy
EMIM.LON 1Y annualized volatility: 17.10% (GHF-based)
Processing IBM...
Using cached IBM copy
Using cached GBPCHF copy
IBM 1Y annualized volatility: 31.64% (GHF-based)
Processing ULVR.LON...
Using cached ULVR.LON copy
Using cached GBPCHF copy
ULVR.LON 1Y annualized volatility: 18.19% (GHF-based)
